# YOLOR — train 5 YOLOv11x detectors that don't forget COCO

This notebook walks the **whole YOLOR training pipeline** end-to-end:
preparing the four custom datasets, training each of the four single-domain models plus the combined YOLOR model, and evaluating all five for COCO retention + custom-class accuracy.

If you've never seen this codebase before, **just run the cells top-to-bottom**. Each section explains what it does and why; you don't need prior knowledge of the data or the design.

**What you'll need**
- A CUDA GPU (≥ 12 GB; 32 GB is comfortable).
- Python 3.9+.
- The four source datasets in standard YOLO layout (see *Verify source datasets* below).
- About **3–5 days** of compute to train all five models on a single V100-class GPU.

## What is YOLOR?

YOLOR is a family of YOLOv11x object detectors used in **camera-primed 6G mmWave beamforming** research (SECON). The detectors look at a camera image and identify *both* general objects (people, cars, etc.) *and* the RF infrastructure that drives beam selection (radios, 5G base stations, lamp-mounted streetlights, mmWave radios). A single network locates everything in one forward pass.

## The catastrophic-forgetting problem

YOLOv11x is pretrained on COCO — 80 classes covering common objects. We want to teach it new classes (radio, streetlight, …) **without losing its COCO knowledge**. Naively fine-tuning on a small custom dataset that only labels the new class will silently destroy COCO performance, for two reasons:

1. **Head truncation** — training with `nc=1` (only `radio`) rebuilds the detection head and discards the 80 pretrained COCO class weights.
2. **Background suppression** — every COCO object in your custom images (the chair, the laptop, the person in the background) is *unlabelled*, so the model learns *"chair = background, laptop = background, person = background"* and actively un-trains those classes.

YOLOR solves this with three ideas working together:

- **Keep `nc = 80 + custom`** with COCO at indices `0..79` and custom classes from `80` up. The pretrained head weights for COCO survive.
- **P1 pseudo-labelling** — run the stock YOLOv11x on every custom image and *merge* its high-confidence COCO detections into the label file as ground truth. Manual labels stay authoritative; a teacher box is dropped if it overlaps a manual box at IoU > 0.5. Now every COCO object in your custom images has positive supervision again.
- **COCO replay** — mix 8,000 actual COCO `train2017` images into each epoch so COCO classes keep seeing real, hand-labelled positives, not just pseudo-labels.

## The five models

| Id | Name | Custom classes (idx) | Source dataset | Train frames |
|---:|------|----------------------|----------------|-------------:|
| 1 | `YOLOR-radio` | radio (80) | IndoorCOTSDataset | 3,599 |
| 2 | `YOLOR-5GBS` | 5G BS (80), LampPost (81) | OutdoorDataset (labeled subset) | 4,107 |
| 3 | `YOLOR-comm-mmWave` | radio (80), mmWave radio (81) | IndoorCommercialDataset (dedup) | ~1,631 |
| 4 | `YOLOR-Streetlights` | streetlight (80) | Streetlights_noAug_fullDataset | 1,498 |
| 5 | `YOLOR` | all five (80–84) | union of the four | ~10,800 |

Models 1–4 are single-domain (one source dataset each). Model 5 is the combined release: it joins all five custom classes into one head. Train them roughly in order — model 5 builds on what the first four validate.

For per-model details, see each model card on Hugging Face.

## 1. Setup

Install dependencies and check your environment. Run this once per machine.

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Pin ultralytics to a known-good version
try:
    import ultralytics
    print(f'ultralytics: {ultralytics.__version__}')
except ImportError:
    pip('ultralytics==8.3.158', 'torch', 'torchvision', 'opencv-python', 'pyyaml', 'pillow')
    import ultralytics
    print(f'installed ultralytics {ultralytics.__version__}')

import torch
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}   memory: {p.total_memory/1e9:.1f} GB')
else:
    print('\n⚠️  No CUDA GPU detected — training will fall back to CPU and be impractically slow.')

## 2. Configure paths

The pipeline auto-detects your project root. Run the cell below to see what it picked and where outputs will go.

If auto-detection picks the wrong root, set the environment variable `YOLOR_ROOT` to the directory that contains `FilteredTrainingDatasets/` *before* importing `cp_config`.

In [ ]:
import cp_config as cfg
cfg.summary()

## 3. Get the data (one-time, after obtaining the datasets from the authors)

The datasets are not publicly redistributed; contact the paper authors for access. Each archive, one per source domain. Each archive unpacks into the layout:

```
<your_path>/<DatasetName>/
├── FullDataset/
│   ├── Images/    (all paired image files, flat)
│   └── Labels/    (all paired YOLO label files, flat)
└── splits/
    ├── train.txt  (one stem per line — the paper's train split)
    ├── val.txt
    └── test.txt
```

**Just point `cp_data.discover_and_materialize()` at the parent folder** and it finds every dataset that has `FullDataset/` + `splits/` and materializes `train/`, `val/`, `test/` directories as **symlinks** (no data duplication). Default path is `<YOLOR_ROOT>/FilteredTrainingDatasets/` — pass an explicit path if your unpacked data lives elsewhere.


In [ ]:
import cp_data
# Auto-discover: scan a folder for any dataset subfolders that have
# `FullDataset/` + `splits/` and materialize `train/val/test/` for each.
cp_data.discover_and_materialize(
    datasets_root=None,    # None = use <YOLOR_ROOT>/FilteredTrainingDatasets/; or pass an explicit Path
    mode='symlink',        # use 'copy' if your filesystem rejects symlinks
)

## 4. Verify the source datasets

Each of the four datasets should live under `<PROJECT_ROOT>/FilteredTrainingDatasets/` in standard YOLO layout (`train/images/` + `train/labels/`, similarly for `val` or `valid`, and `test`).

The cell below checks each dataset is present and reports the per-split frame counts. If anything is missing, see the *Dataset placement* section in `../README.md`.

In [ ]:
for key, path in cfg.DATASETS.items():
    val_split = cfg.SPLIT_DIRNAMES[key]  # 'val' for most, 'valid' for streetlight
    flag = '✅' if path.exists() else '❌ MISSING'
    def count(sp):
        d = path/sp/'images'
        return len(list(d.iterdir())) if d.exists() else 0
    print(f'  {flag}  {key:11s}  train={count("train"):5d}  {val_split}={count(val_split):4d}  test={count("test"):4d}   @ {path}')

## 5. Stage COCO

YOLOR uses COCO twice:
- **`val2017`** (5,000 images) — for measuring how much COCO performance the finetuned model retains vs the stock baseline.
- **A class-stratified 8,000-image slice of `train2017`** — mixed into each training epoch as *replay* so COCO classes keep seeing real positives.

The cell below downloads, converts, and slices COCO. On a re-run it's a no-op.

> ⚠️ COCO val2017 is ~6 GB; train2017 is ~19 GB. Make sure you have disk space.

In [ ]:
!python cp_coco_replay.py --step coco

## 6. Prepare data for each model

For each model we:

1. **Pseudo-label** the custom dataset — run stock YOLOv11x over the custom images and merge its high-confidence COCO detections into a sibling `labels_p1/` directory. Manual labels stay authoritative.
2. **(Commercial only)** Perceptual-hash dedup. The IndoorCommercialDataset is ~14k burst-video frames; we keep the ~1,631 perceptually distinct ones (`--threshold 1`) to avoid overtraining on near-duplicates.
3. **Materialize** — assemble a per-model `data.yaml` that points at the custom train/val/test splits + the COCO replay slice + COCO val2017. This is what training reads.

Each cell below is independent — you can run any subset of models.

In [ ]:
# Model 1: YOLOR-radio (IndoorCOTSDataset)
!python cp_pseudolabel.py --dataset cots --splits train val test --conf 0.5 --iou-drop 0.5
!python cp_coco_replay.py --step materialize --model 1

In [ ]:
# Model 2: YOLOR-5GBS (OutdoorDataset, labeled subset only)
!python cp_pseudolabel.py --dataset outdoor --splits train val test --conf 0.5 --iou-drop 0.5
!python cp_coco_replay.py --step materialize --model 2

In [ ]:
# Model 3: YOLOR-comm-mmWave (IndoorCommercialDataset, dedup th=1)
# Commercial source = ~14,386 burst-video frames. dHash threshold=1 keeps only
# perceptually distinct frames -> ~1,631 diverse images for training.
!python cp_pseudolabel.py --dataset commercial --splits train val test --conf 0.5 --iou-drop 0.5
!python cp_dedup.py --dataset commercial --splits train --threshold 1
!python cp_coco_replay.py --step materialize --model 3 --dedup-splits train

In [ ]:
# Model 4: YOLOR-Streetlights (Streetlights_noAug_fullDataset)
# Uses the 'valid' split name (not 'val') — that's how this dataset was packaged.
!python cp_pseudolabel.py --dataset streetlight --splits train valid test --conf 0.5 --iou-drop 0.5
!python cp_coco_replay.py --step materialize --model 4

In [ ]:
# Model 5: YOLOR (unified, all 5 custom classes — radio, 5G BS, LampPost, mmWave radio, streetlight)
# Requires the per-dataset prep cells above to have been run (their `labels_p1/<dataset>/` are the source).
# Translates each dataset's labels to the unified class scheme (cots:80->80, outdoor:80->81/81->82,
# commercial:80->80/81->83, streetlight:80->84), then materializes the union as model 5.
for ds in ('cots', 'outdoor', 'commercial', 'streetlight'):
    !python cp_data.py --action translate_unified --dataset {ds}
!python cp_coco_replay.py --step materialize --model 5 --dedup-splits train

## 7. Train

`cp_train.py` is standard Ultralytics fine-tuning + a lightweight 2-hour status logger. **No fragile metric callbacks** — evaluation and final-model selection are intentionally separated out into `cp_eval.py` (next section), which runs as its own Python process. This decoupling is deliberate; it sidesteps a subtle Ultralytics quirk in which per-class mAP can read 0 for some custom classes when computed inside the training process, even though the model is excellent. Standalone eval gets the right numbers.

**Hyperparameters** are locked in `cp_config.TRAIN_HYP` (200 epochs, `cos_lr`, `close_mosaic=20`, `lr0=0.01`, `mosaic=1.0`, `mixup=0.1`, `save_period=5`). Override per run via env vars: e.g. `YOLOR_BATCH=8 python cp_train.py …` or `YOLOR_EPOCHS=300 python cp_train.py …`.

Without `--run`, the script does a **1-epoch timing probe + ETA only** — a safe way to verify everything is wired correctly before committing to a multi-day run. Add `--run` for full training.

**Resume-aware:** if training is interrupted, re-run the same cell. `cp_train.py` auto-resumes from `runs/<model_name>/weights/last.pt`. Add `--fresh` to force a from-scratch restart.

**Expected wall-clock per model on a single V100-32GB at the default 200 epochs:** ~24–36 h. Faster GPUs scale roughly proportionally.

In [ ]:
# Optional: safe 1-epoch probe (prints ETA, doesn't start the long run)
!python cp_train.py --model 1

In [ ]:
# Train model 1: YOLOR-radio
!python cp_train.py --model 1 --run

In [ ]:
# Train model 2: YOLOR-5GBS
!python cp_train.py --model 2 --run

In [ ]:
# Train model 3: YOLOR-comm-mmWave
!python cp_train.py --model 3 --run

In [ ]:
# Train model 4: YOLOR-Streetlights
!python cp_train.py --model 4 --run

In [ ]:
# Train model 5: YOLOR (combined release model)
# NOTE: requires the combined data build for model 5 to be in place first;
# see cp_data.py docstring for the group-aware merged-split prep step.
!python cp_train.py --model 5 --run

## 8. Evaluate (this is the source of truth for paper numbers)

`cp_eval.py` is a *standalone* evaluator that runs in a separate Python process from training. It reports, per model:

- **COCO retention vs stock** — per-class mAP50-95 on COCO `val2017`, finetuned vs stock `yolo11x`, with a ratio (`1.0` = no forgetting).
- **Custom-class mAP50-95** — on the model's own custom test split (no COCO mixed in).

Outputs:
- `outputs/<model_name>_eval.csv` — per-class table (suitable for a paper appendix).
- A printed Markdown summary in the cell output, ready to paste into a paper table.

> Note: `cp_eval` uses `runs/<model_name>/weights/last.pt` (the converged final checkpoint). It deliberately ignores `best.pt` — Ultralytics' default fitness pick is COCO-biased here and tends to be a near-stock early epoch.

In [ ]:
!python cp_eval.py --model 1

In [ ]:
!python cp_eval.py --model 2

In [ ]:
!python cp_eval.py --model 3

In [ ]:
!python cp_eval.py --model 4

In [ ]:
!python cp_eval.py --model 5

## 9. Going further

**Cluster (SLURM) training.** Template SLURM scripts live in `code/slurm/`. See `code/slurm/README.md` for what to change to adapt them to a cluster other than the one this project was developed on (HCC Swan + NRDStor). On a single dedicated GPU box you don't need SLURM at all — the cells above are the whole pipeline.

**Add your own dataset.**

1. Drop a new dataset in standard YOLO layout under `<PROJECT_ROOT>/FilteredTrainingDatasets/<YourDataset>/`.
2. Add an entry to `cp_config.DATASETS` pointing to it.
3. Add a `cp_config.MANUAL_REMAP` entry mapping the source class ids to the unified YOLOR scheme (COCO at `0..79`; your custom class at index `80+`).
4. Add a `cp_config.CUSTOM_BY_MODEL` entry for the new model id.
5. Run the same notebook cells with the new model id.

**Change the recipe.** All hyperparameters live in `cp_config.TRAIN_HYP`. Each has a `CP_*` env override for per-run experimentation (e.g. `YOLOR_EPOCHS`, `YOLOR_BATCH`, `YOLOR_IMGSZ`, `YOLOR_MIXUP`, `YOLOR_REPLAY_N`).

## Citation

If you use this code, please cite the associated paper:

> *Camera-primed 6G mmWave Beamforming* (SECON). See the paper PDF in the project root for the full reference and BibTeX.